In [ ]:
%%capture
import os
import pandas as pd
import numpy as np
from dj_notebook import activate
from django_pandas.io import read_frame
from pathlib import Path

env_file = os.environ["INTECOMM_ENV"]
documents_folder = Path(os.environ["INTECOMM_DOCUMENTS_FOLDER"])
plus = activate(dotenv_file=env_file)
report_folder = Path(documents_folder)


In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858
from edc_pdutils.dataframes import get_crf, get_subject_visit, get_appointments
from tabulate import tabulate

df_main = get_df_main_1858(None)
# df_main = pd.read_csv(Path("/Users/erikvw/Documents/ucl/protocols/intecomm/analysis/primary/") / "df_main_1858.csv")


In [ ]:

def get_cells_for_continuous_var(df)->list[str]:
    """ From describe(), format 3 cells as:

        +======================+
        | 930                  |
        +----------------------+
        | 127.69(16.84)        |
        +----------------------+
        | 127.00(82.00–183.00) |
        +----------------------+
    """
    return [
        f"{int(df['count'])}",
        f"{df['mean']:.2f}({df['std']:.2f})",
        f"{df['50%']:.2f}({df['min']:.2f}–{df['max']:.2f})"
    ]


def get_formatted_rows(df, label):
    """Returns 5 columns"""
    baseline_a = df[df['assignment'] == 'a'][f'bp_{label}_baseline'].describe()
    endline_a = df[df['assignment'] == 'a'][f'bp_{label}_endline'].describe()
    baseline_b = df[df['assignment'] == 'b'][f'bp_{label}_baseline'].describe()
    endline_b = df[df['assignment'] == 'b'][f'bp_{label}_endline'].describe()
    baseline_all = df[f'bp_{label}_baseline'].describe()
    endline_all = df[f'bp_{label}_endline'].describe()

    return  {
    'Timepoint': ['Baseline', '', '', 'Endline', '', ''],
    'Statistics': ['n', 'Mean(sd)', 'Median(min-max)','n', 'Mean(sd)', 'Median(min-max)'],
    'Treatment A': [
        *get_cells_for_continuous_var(baseline_a),
        *get_cells_for_continuous_var(endline_a),
    ],
    'Treatment B': [
        *get_cells_for_continuous_var(baseline_b),
        *get_cells_for_continuous_var(endline_b),
    ],
    'All': [
        *get_cells_for_continuous_var(baseline_all),
        *get_cells_for_continuous_var(endline_all),
    ],
}

# all subjects
df1 = df_main.copy()
sys_table = {'Condition': ['All', '', '', '', '', '']}
dia_table = {'Condition': ['All', '', '', '', '', '']}
sys_table.update({
    'Parameter': ['Blood pressure: systolic (mmHg)', '', '', '', '', ''], **get_formatted_rows(df1, "sys")
})
dia_table.update({
    'Parameter': ['Blood pressure: diastolic (mmHg)', '', '', '', '', ''], **get_formatted_rows(df1, "dia")
})
sys_df = pd.DataFrame(sys_table)
dia_df = pd.DataFrame(dia_table)

# ncd only subjects
df1 = df1[(df1.ncd==1) & (df1.hiv==0)].copy()
sys_table = {'Condition': ['NCD', '', '', '', '', '']}
dia_table = {'Condition': ['NCD', '', '', '', '', '']}
sys_table.update({
    'Parameter': ['Blood pressure: systolic (mmHg)', '', '', '', '', ''], **get_formatted_rows(df1, "sys")
})
dia_table.update({
    'Parameter': ['Blood pressure: diastolic (mmHg)', '', '', '', '', ''], **get_formatted_rows(df1, "dia")
})
sys_ncd_df = pd.DataFrame(sys_table)
dia_ncd_df = pd.DataFrame(dia_table)

# concat results
summary_df = pd.concat([sys_df, dia_df, sys_ncd_df, dia_ncd_df], ignore_index=True)
# generate table
table = tabulate(summary_df, headers='keys', tablefmt='grid')

# Write the table to file
documents_folder = Path(os.environ["INTECOMM_DOCUMENTS_FOLDER"])
path = documents_folder / 'summary.txt'
with open(path, 'w') as file:
    file.write(table)


In [ ]:
path = documents_folder / 'summary.csv'
summary_df.to_csv(path_or_buf=path, index=False)

In [ ]:
df_main[(df_main.bp_sys_baseline.notna()) & (df_main.bp_sys_endline.notna())].groupby("assignment")["bp_sys_endline"].describe()


In [ ]:
grouped_endline = df_main[df_main.bp_visit_code_endline==1120.0].groupby("assignment").size()
total_endline = df_main[df_main.bp_visit_code_endline==1120.0].subject_identifier.count()
print(grouped_endline, f"All {total_endline}")

In [ ]:
df_main[["bp_visit_code_baseline", "bp_visit_code_endline", "bp_sys_baseline", "bp_sys_endline"]]

In [ ]:
df_appointment = get_appointments()


In [ ]:
from edc_appointment.constants import SKIPPED_APPT, NEW_APPT
from scipy.stats import ttest_ind

df_tmp = pd.merge(df_appointment, df_main[["subject_identifier", "assignment"]], how="left", on="subject_identifier")
df_tmp = df_tmp[(df_tmp.visit_code_sequence==0) &
       ~(df_tmp.appt_status==SKIPPED_APPT) &
       ~(df_tmp.appt_status==NEW_APPT)
]
df_tmp = df_tmp.groupby(by=["subject_identifier", "appt_timing", "assignment"]).size().to_frame().reset_index()
df_tmp = df_tmp.pivot_table(index=["subject_identifier","assignment"], columns=["appt_timing"], values=0).fillna(0).astype(int).reset_index()
df_tmp["total_appts"] = df_tmp["missed"] + df_tmp["ontime"]
df_tmp["prop_missed"] = df_tmp["missed"] / df_tmp["total_appts"]

t_stat, p_value = ttest_ind(df_tmp[df_tmp.assignment=="a"]["prop_missed"], df_tmp[df_tmp.assignment=="b"]["prop_missed"], equal_var=False)
t_stat, p_value


In [ ]:
df_visit = get_subject_visit(model="intecomm_subject.subjectvisit")
df_visit.reason.value_counts()

In [ ]:
df_tmp = df_visit[(df_visit.visit_code_sequence==0) & (df_visit.reason!="missed")].groupby(by=["subject_identifier", "baseline_datetime", "last_visit_datetime"]).size().to_frame().reset_index()
df_tmp.columns = ["subject_identifier", "baseline_datetime", "last_visit_datetime", "attended_visits"]
df_tmp

In [ ]:
df_main = df_main.merge(df_tmp, how="left", on="subject_identifier", suffixes=("", "_y"))

In [ ]:
df_main

In [ ]:
df_vitals = get_crf(model="intecomm_subject.vitals", subject_visit_model="intecomm_subject.subjectvisit")


In [ ]:
# Where two measurements are not available or possible, a single measurement will be used.
def get_dia_avg(s):
    new_avg = np.nan
    if pd.notna(s["dia_blood_pressure_one"]) and pd.notna(s["dia_blood_pressure_two"]):
        new_avg =  (s["dia_blood_pressure_one"] + s["dia_blood_pressure_two"]) / 2
    elif pd.notna(s["dia_blood_pressure_one"]) and pd.isna(s["dia_blood_pressure_two"]):
        new_avg = s["dia_blood_pressure_one"]
    return new_avg

def get_sys_avg(s):
    new_avg = np.nan
    if pd.notna(s["sys_blood_pressure_one"]) and pd.notna(s["sys_blood_pressure_two"]):
        new_avg =  (s["sys_blood_pressure_one"] + s["sys_blood_pressure_two"]) / 2
    elif pd.notna(s["sys_blood_pressure_one"]) and pd.isna(s["sys_blood_pressure_two"]):
        new_avg = s["sys_blood_pressure_one"]
    return new_avg


df_vitals["bp_dia_avg"] = df_vitals.apply(get_dia_avg, axis=1)
df_vitals["bp_sys_avg"] = df_vitals.apply(get_sys_avg, axis=1)


In [ ]:
df_vitals["check_dia"] = df_vitals["bp_dia_avg"] == df_vitals["dia_blood_pressure_avg"]

In [ ]:
df_vitals["check_sys"] = df_vitals["bp_sys_avg"] == df_vitals["sys_blood_pressure_avg"]

In [ ]:
df_vitals.check_dia.value_counts()

In [ ]:
df_vitals.check_sys.value_counts()


In [ ]:
columns = ["subject_identifier", "visit_code", ]
df_tmp = df_vitals.copy()
df_bp_baseline = df_tmp[df_tmp.visit_code==1000.0][["subject_identifier","visit_datetime", "bp_dia_avg", "bp_sys_avg"]]
df_bp_baseline = df_bp_baseline.rename(columns={
    "visit_datetime": "baseline_datetime",
    "bp_dia_avg":"baseline_bp_dia_avg",
    "bp_sys_avg":"baseline_bp_sys_avg"}
)

df_bp_endline = df_tmp[["subject_identifier","visit_datetime", "bp_dia_avg", "bp_sys_avg"]]
df_bp_endline = df_bp_endline.rename(columns={"visit_datetime": "last_visit_datetime", "bp_dia_avg":"endline_bp_dia_avg", "bp_sys_avg":"endline_bp_sys_avg"})


df_main = df_main.merge(df_bp_baseline, how="left", on=["subject_identifier","baseline_datetime"], suffixes=("", "_y"))
df_main = df_main.merge(df_bp_endline, how="left", on=["subject_identifier","last_visit_datetime"], suffixes=("", "_y"))

df_main.groupby(by=["assignment"]).agg({"baseline_bp_sys_avg":"mean", "endline_bp_sys_avg":"mean","baseline_bp_dia_avg":"mean", "endline_bp_dia_avg":"mean"})


In [ ]:
# categorize controlled, not comtrolled, severe
def get_bp_controlled(s, word):
    if pd.isna(s[f"{word}_bp_dia_avg"]) or pd.isna(s[f"{word}_bp_sys_avg"]):
        controlled = False if s["assignment"] == "a" else True
    elif s[f"{word}_bp_dia_avg"] < 90.0 and s[f"{word}_bp_sys_avg"] < 140.0:
        controlled = True
    else:
        controlled = False
    return controlled

def get_baseline_bp_controlled(s):
    return get_bp_controlled(s, "baseline")

def get_endline_bp_controlled(s):
    return get_bp_controlled(s, "endline")

df_main["baseline_bp_controlled"] = df_main.apply(get_baseline_bp_controlled, axis=1)
df_main["endline_bp_controlled"] = df_main.apply(get_endline_bp_controlled, axis=1)

In [ ]:
df_main.baseline_bp_controlled.value_counts()
df_main.groupby(by=["assignment", "baseline_bp_controlled"]).size()


In [ ]:
df_main.endline_bp_controlled.value_counts()
df_main.groupby(by=["assignment", "endline_bp_controlled"]).size()


In [ ]:
def get_bp_controlled(s):
    if pd.isna(s["baseline_bp_dia_avg"]) or pd.isna(s["baseline_bp_sys_avg"]):
        controlled = False if s["assignment"] == "a" else True
    elif s["baseline_bp_dia_avg"] < 90.0 and s["baseline_bp_sys_avg"] < 140.0:
        controlled = True
    else:
        controlled = False
    return controlled




In [ ]:
df_main

In [ ]:
df_vitals[df_vitals.check==False][["check", "baseline_bp_dia_avg", "dia_blood_pressure_avg", "dia_blood_pressure_one", "dia_blood_pressure_two"]]

In [ ]:
df_tmp = df_vitals.copy()
df_tmp= df_tmp.rename(columns={
    "dia_blood_pressure_avg": "baseline_bp_dia_avg",
    "sys_blood_pressure_avg":"baseline_bp_sys_avg"
})


In [ ]:
df_bp_baseline = df_vitals[df_vitals.visit_code==1000.0][columns]

In [ ]:
df_main.columns